# MedSAM2: segment a 3D CT by prompting a single slice

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rekalantar/MedSAM2-3D-CT/blob/main/tutorial/medsam2_3d_ct.ipynb)

SAM 2 tracks objects across video frames using memory attention. A CT volume has the
same structure — consecutive slices barely differ — so one box on one slice can be
propagated through the whole stack.

This notebook is **self-contained**: every function it uses is defined here. The only
external install is MedSAM2 itself.

**Runtime → Change runtime type → T4 GPU** before running anything.

---

**This notebook runs the method. [The article](ARTICLE_URL) explains it** — what each step
does and why, and what a bounding box does and doesn't tell the model.

## 1 · Setup

In [ ]:
import torch

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU, then rerun."
print(torch.cuda.get_device_name(0))

In [ ]:
%%capture
!pip install -q SimpleITK imageio huggingface_hub
!git clone -q https://github.com/bowang-lab/MedSAM2.git /content/MedSAM2
%cd /content/MedSAM2
!pip install -q -e ".[dev]"
!bash download.sh

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk
import PIL.Image
from huggingface_hub import hf_hub_download
from scipy.ndimage import binary_erosion, generate_binary_structure, label

CKPT = "/content/MedSAM2/checkpoints/MedSAM2_latest.pt"
CONFIG = "configs/sam2.1_hiera_t512.yaml"
DATASET = "wanglab/CT_DeepLesion-MedSAM2"
ROTATE = 1        # clockwise quarter-turns, display only — these volumes read sideways

assert os.path.exists(CKPT), "download.sh did not produce MedSAM2_latest.pt"
print(f"checkpoint {os.path.getsize(CKPT) / 1e6:.0f} MB")

## 2 · Helpers

Loading, scoring and drawing. Skim this — the interesting part is section 4.

In [ ]:
def load_nifti(path):
    """Return (z, y, x) array and voxel spacing in mm, also (z, y, x).

    SimpleITK reports spacing as (x, y, z), so it is reversed to match the array axes.
    Getting this backwards silently corrupts any physical measurement downstream.
    """
    image = sitk.ReadImage(path)
    return sitk.GetArrayFromImage(image), tuple(reversed(image.GetSpacing()))


def dice(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    total = a.sum() + b.sum()
    return 1.0 if total == 0 else 2 * (a & b).sum() / total


def rotate_cw(stack, turns=1):
    """Rotate the in-plane axes for display. Apply to volume and mask together."""
    return stack if turns % 4 == 0 else np.rot90(stack, k=-turns, axes=(1, 2))

In [ ]:
CYAN, CORAL = (63, 193, 201), (255, 107, 91)


def overlay(slice_u8, mask, color=CYAN, alpha=0.4):
    rgb = np.stack([slice_u8] * 3, axis=-1).astype(np.float32)
    mask = np.asarray(mask, bool)
    if mask.any():
        rgb[mask] = (1 - alpha) * rgb[mask] + alpha * np.array(color, np.float32)
        edge = mask ^ binary_erosion(mask, generate_binary_structure(2, 1))
        rgb[edge] = color
    return np.clip(rgb, 0, 255).astype(np.uint8)


def save_gif(volume_u8, masks, path, fps=4, rotate=0):
    import imageio.v2 as imageio
    volume_u8, masks = rotate_cw(volume_u8, rotate), rotate_cw(masks, rotate)
    frames = [overlay(volume_u8[i], masks[i]) for i in range(len(volume_u8))]
    imageio.mimsave(path, frames, fps=fps, loop=0)
    return path


def plot_slices(volume_u8, masks, truth=None, rotate=0, pad=40, ncols=7, title=None):
    """Contact sheet, cropped to the masked region. Truth cyan, prediction coral."""
    region = masks if truth is None else (masks | truth)
    idx = region.any(axis=(1, 2)).nonzero()[0]
    ys, xs = np.where(region.any(axis=0))
    y0, y1 = max(0, ys.min() - pad), min(region.shape[1], ys.max() + pad + 1)
    x0, x1 = max(0, xs.min() - pad), min(region.shape[2], xs.max() + pad + 1)

    nrows = int(np.ceil(len(idx) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.2 * ncols, 2.5 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")

    def crop(plane):
        return rotate_cw(plane[None, y0:y1, x0:x1], rotate)[0]

    for n, i in enumerate(idx):
        ax = axes.flat[n]
        ax.imshow(crop(volume_u8[i]), cmap="gray")
        if truth is not None:
            ax.contour(crop(truth[i]), levels=[0.5],
                       colors=[np.array(CYAN) / 255], linewidths=1.4)
        ax.contour(crop(masks[i]), levels=[0.5],
                   colors=[np.array(CORAL) / 255], linewidths=1.4)
        ax.set_title(f"slice {i}", fontsize=9)
    if title:
        fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    return fig

## 3 · Load a case

The MedSAM2 demo set ships per-case volumes with ground-truth masks, a few MB each.

**Windowing.** A CT stores Hounsfield units from about −1000 (air) past +1000 (bone).
The network takes 8-bit — 256 levels. Scale the full range naively and abdominal soft
tissue, a band only ~150 units wide, collapses into a handful of near-identical greys.
So keep a window and discard the rest: width 400 centred at 40.

**The prompt** comes from the ground-truth label here — standard practice for evaluating
promptable models, since it simulates a perfect user box and isolates propagation quality.

In [ ]:
def window_hu(volume, width=400, level=40):
    """Hounsfield window -> uint8."""
    lo, hi = level - width / 2, level + width / 2
    return ((np.clip(volume, lo, hi) - lo) / (hi - lo) * 255).astype(np.uint8)


def load_case(case, margin=5):
    image = hf_hub_download(DATASET, f"images/{case}_0000.nii.gz", repo_type="dataset")
    label = hf_hub_download(DATASET, f"labels/{case}.nii.gz", repo_type="dataset")

    volume_hu, spacing = load_nifti(image)
    truth = load_nifti(label)[0] > 0

    key = int(truth.sum(axis=(1, 2)).argmax())      # largest cross-section
    ys, xs = np.where(truth[key])
    box = [int(xs.min()) - margin, int(ys.min()) - margin,
           int(xs.max()) + margin, int(ys.max()) + margin]

    return dict(name=case, hu=volume_hu, volume=window_hu(volume_hu), truth=truth,
                spacing=spacing, key=key, box=box,
                span=int(truth.any(axis=(1, 2)).sum()))


case = load_case("000009_03_01_036-048")
print(f"volume {case['volume'].shape}  spacing {tuple(round(s, 2) for s in case['spacing'])}")
print(f"HU     {case['hu'].min():.0f} to {case['hu'].max():.0f}")
print(f"lesion {case['span']} slices, largest at {case['key']}, box {case['box']}")

## 4 · The pipeline

Four steps, and this is the whole method.

### Step 1 — shape the volume like a video

SAM 2's image encoder is a natural-image backbone. It wants three channels at 512×512
with ImageNet normalisation, as a float tensor. A CT slice is one channel at whatever
resolution the scanner produced. So each slice is resized, copied across three channels
and normalised — producing a tensor shaped exactly like a batch of video frames.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def to_frames(volume_u8, size=512):
    """(D, H, W) uint8 -> (D, 3, size, size) normalised float32 on the GPU."""
    rgb = np.zeros((len(volume_u8), 3, size, size), dtype=np.uint8)
    for i, plane in enumerate(volume_u8):
        resized = PIL.Image.fromarray(plane).resize((size, size), PIL.Image.BILINEAR)
        rgb[i] = np.asarray(resized)[None].repeat(3, axis=0)

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    t = torch.from_numpy(rgb).to(dev).float() / 255.0
    t -= torch.tensor(IMAGENET_MEAN, device=dev).view(1, 3, 1, 1)
    t /= torch.tensor(IMAGENET_STD, device=dev).view(1, 3, 1, 1)
    return t


frames = to_frames(case["volume"])
print(f"{case['volume'].shape} uint8  ->  {tuple(frames.shape)} {frames.dtype}")

### Step 2 — open an inference state

The inference state is SAM 2's working memory for one volume: per-slice image features,
plus the representation of whatever object you prompt.

Note the dimensions passed in are the volume's **original** height and width, not the
512×512 it was just resized to. That tells the predictor which coordinate system your
prompt is in, and what resolution to return masks at. Rescaling your box to 512-space
instead runs without error and segments the wrong region.

In [ ]:
from sam2.build_sam import build_sam2_video_predictor_npz

predictor = build_sam2_video_predictor_npz(CONFIG, CKPT)

depth, height, width = case["volume"].shape
state = predictor.init_state(frames, height, width)   # original dims, not 512
print(f"state open for {depth} slices at {height}x{width}")

### Step 3 — place one box

One rectangle on one slice. That is the entire human contribution.

A box says *the object is somewhere inside here*. It says nothing about the boundary,
the texture or the shape — the model infers all of that from the image.

In [ ]:
predictor.add_new_points_or_box(
    inference_state=state,
    frame_idx=case["key"],                                  # the slice being prompted
    obj_id=1,                                               # track this as object 1
    box=np.asarray(case["box"], dtype=np.float32),          # original pixel coords
)
print(f"prompted slice {case['key']} with box {case['box']}")

### Step 4 — propagate, both directions

Memory attention does the work: starting from the prompted slice, the model segments
the next one using its image features *and* its memory of what it just found; that
result updates the memory and it moves on.

Then run it again in reverse. The prompted slice sits in the **middle** of the lesion,
so a forward-only sweep segments one half and stops.

In [ ]:
masks = np.zeros(case["volume"].shape, dtype=bool)

for reverse in (False, True):
    for idx, _ids, logits in predictor.propagate_in_video(state, reverse=reverse):
        masks[idx] = (logits[0] > 0).cpu().numpy().squeeze()


def largest_component(mask):
    """Drop propagation leakage. Assumes a single connected object."""
    labelled, count = label(mask)
    if count <= 1:
        return mask
    sizes = np.bincount(labelled.ravel())
    sizes[0] = 0
    return labelled == sizes.argmax()


masks = largest_component(masks)
print(f"prediction    {masks.sum():>7,} voxels, {masks.any(axis=(1,2)).sum()} slices")
truth = case['truth']
print(f"ground truth  {truth.sum():>7,} voxels, {truth.any(axis=(1,2)).sum()} slices")

## 5 · The result

In [ ]:
print(f"Dice                   {dice(truth, masks):.3f}")
print(f"Dice on the key slice  {dice(truth[case['key']], masks[case['key']]):.3f}")

In [ ]:
z = (masks | truth).any(axis=(1, 2)).nonzero()[0]
lo, hi = max(0, z.min() - 2), min(len(case["volume"]), z.max() + 3)

save_gif(case["volume"][lo:hi], masks[lo:hi], "propagation.gif", fps=4, rotate=ROTATE)
print(f"{os.path.getsize('propagation.gif') / 1e6:.2f} MB, {hi - lo} frames")

In [ ]:
import IPython.display

IPython.display.Image("propagation.gif")

In [ ]:
fig = plot_slices(case["volume"], masks, truth, rotate=ROTATE,
                  title=f"{case['name']} \u2014 Dice {dice(truth, masks):.3f}")
fig.savefig("slices.png", dpi=150, bbox_inches="tight")

## 6 · Where it struggles

**One prompt segments one object.** Memory attention tracks the thing you pointed at;
it has no mechanism for discovering a second structure elsewhere in the volume. If your
label marks several lesions, prompt each one separately and skip `largest_component` —
which by design keeps only the biggest blob and deletes the rest.

**Accuracy is highest near the prompt.** Per-slice Dice against distance from the
prompted slice shows where the cost actually falls.

In [ ]:
def per_slice_dice(truth, pred, key):
    idx = truth.any(axis=(1, 2)).nonzero()[0]
    return idx - key, np.array([dice(truth[i], pred[i]) for i in idx])


offsets, scores = per_slice_dice(truth, masks, case["key"])
for off, d in zip(offsets, scores):
    print(f"  {off:+3d}  {d:.3f}  {'#' * int(d * 40)}")

plt.figure(figsize=(8, 3.5))
plt.plot(offsets, scores, "o-", color="#3FC1C9", linewidth=2)
plt.axvline(0, color="#FF6B5B", linestyle="--", label="prompted slice")
plt.xlabel("slices from prompt"); plt.ylabel("Dice")
plt.ylim(0, 1.05); plt.legend(); plt.tight_layout()
plt.savefig("decay.png", dpi=150, bbox_inches="tight")

---

Project: https://github.com/rekalantar/MedSAM2-3D-CT

The same functions are available as an installable package in that repo if you would
rather import them than copy them.